## Data Profiling

In [ ]:
import pandas as pd
import os 
import seaborn as sns 
import matplotlib.pyplot as plt

df_typed = pd.read_parquet("../data/bronze/clean_dataset.parquet",  engine='fastparquet', index=False)
df_typed

In [ ]:
data = df_typed.copy()
data = data.sort_values(["date", "time"]).reset_index(drop=True)

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
import numpy as np

stats = data.describe(include='all')
stats.loc['dtype'] = data.dtypes
stats.loc['rows_dataset'] = len(data)

stats.loc['n_missing'] = data.isna().sum()
stats.loc['% missing'] = data.isna().sum() * 100

numeric_cols = data.select_dtypes(include=np.number).columns
if len(numeric_cols) > 0:
    stats.loc['skew', numeric_cols] = data[numeric_cols].skew()
    stats.loc['kurtosis', numeric_cols] = data[numeric_cols].kurtosis()
    stats.loc['range', numeric_cols] = data[numeric_cols].max() - data[numeric_cols].min()
    stats.loc['iqr', numeric_cols] = data[numeric_cols].quantile(0.75) - data[numeric_cols].quantile(0.25)

stats

## Univariate Analysis


### date


What is the distribution of the top 20 records per date, and are there any anomalies or patterns in the frequency of dates?


In [ ]:
data['date_length'] = data['date'].str.len()
print("\nUnique lengths of date strings:", data['date_length'].unique())
print("Counts per length:")
print(data['date_length'].value_counts())

In [ ]:
date_counts = data['date'].value_counts()
print("\nTop 20 most frequent dates:")
date_counts.head(20)


There is no clear pattern here. Almost every month has a day with a really high amount of bookings

In [ ]:
plt.figure(figsize=(12,5))
date_counts.head(20).plot(kind='bar')
plt.title("Top 20 most frequent dates")
plt.xlabel("Date")
plt.ylabel("Count")
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
date_counts.tail(20).plot(kind='bar')
plt.title("Top 20 least frequent dates")
plt.xlabel("Date")
plt.ylabel("Count")
plt.grid()
plt.show()


The range bookings in the top 20 days is of 17 bookings. That is interesting. Why is this the maximum number? Was it coincidence? Do we have a bottleneck somewhere? Is it the local record of the city we are analyzing? 

Every day of the year has between 350 and 460 records aprox

It would be interesting to check if those are special days of the year and which is the relationship with cancellations (the amount is the real maximum demand of the service or that flattening is caused by, for example, some service need of improvement)

In [ ]:
plot_file = 'imgs/date_lineplot.png'

if os.path.exists(plot_file):
    img = plt.imread(plot_file)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')  
    plt.show()
else:
    plt.figure(figsize=(12, 6))
    daily_counts = data.groupby('date').size()
    plt.plot(daily_counts.index, daily_counts.values)

    first_date = daily_counts.index.min()
    last_date = daily_counts.index.max()
    first_date_label = daily_counts.index.min()
    last_date_label = daily_counts.index.max()
    plt.axvline(first_date, color='red', linestyle='--', alpha=0.7, label=first_date_label)
    plt.axvline(last_date, color='green', linestyle='--', alpha=0.7, label=last_date_label)
    plt.title('Total Uber Rides in 2024')
    plt.xlabel('Date')
    plt.ylabel('Number of Rides')

    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(10))
    plt.grid()
    plt.legend()
    plt.tight_layout()
    os.makedirs('imgs', exist_ok=True) 
    plt.savefig(plot_file)
    plt.show()



how can I have 365 unique values if the 31st of Dec does not appear? Maybe there is a 29th of Feb? 

In [ ]:
data[data["date"] == "2024-02-29"].head()

Ok, it's a leap year



### time

In [ ]:
data['time_length'] = data['time'].str.len()
print("\nUnique lengths of time strings:", data['time_length'].unique())
print("Counts per length:")
print(data['time_length'].value_counts())

In [ ]:
time = data['time'].value_counts()
print("\nTop 20 most frequent dates:")
time.head(20)

In [ ]:
plot_file = 'imgs/time_lineplot.png'

if os.path.exists(plot_file):
    img = plt.imread(plot_file)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')  
    plt.show()
else:
    plt.figure(figsize=(12, 6))

    time_counts = data.groupby('time').size().sort_index()
    plt.plot(time_counts.index, time_counts.values)

    first_time = time_counts.index.min()
    last_time = time_counts.index.max()
    plt.axvline(first_time, color='red', linestyle='--', alpha=0.7, label=f'First Time ({first_time})')
    plt.axvline(last_time, color='green', linestyle='--', alpha=0.7, label=f'Last Time ({last_time})')

    plt.title('Uber Rides during the day in 2024')
    plt.xlabel('Time')
    plt.ylabel('Number of Rides')

    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(10))

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    os.makedirs('imgs', exist_ok=True) 
    plt.savefig(plot_file)
    plt.show()


As expected there is so much data because of the seconds that I cannot even see the full lineplot. Let's group by other time units

In [ ]:
plot_file = 'imgs/time_lineplot_hour.png'

if os.path.exists(plot_file):
    img = plt.imread(plot_file)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    plt.figure(figsize=(12, 6))
    data['time'] = pd.to_datetime(data['time'])
    time_counts = (
        data
        .groupby(data['time'].dt.hour)
        .size()
        .sort_index()
    )

    plt.plot(time_counts.index, time_counts.values)

    first_time = time_counts.index.min()
    last_time = time_counts.index.max()
    plt.axvline(first_time, color='red', linestyle='--', alpha=0.7, label=f'First Hour ({first_time}:00)')
    plt.axvline(last_time, color='green', linestyle='--', alpha=0.7, label=f'Last Hour ({last_time}:00)')

    plt.title('Uber Rides during the Day (Hourly)')
    plt.xlabel('Hour of Day')
    plt.ylabel('Number of Rides')

    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    os.makedirs('imgs', exist_ok=True)
    plt.savefig(plot_file)
    plt.show()


The plot shows a clear bimodal daily pattern in Uber ride demand. Activity is minimal during the early morning hours, increases sharply during the morning commute with a peak around 10 AM, dips around midday, and reaches its highest level in the early evening (around 6 PM). Demand then steadily declines toward the end of the day

### new date + time related features

In [ ]:
# the 1st of January of 2024 was Monday

data['datetime'] = pd.to_datetime(data['date'] + " " + data['time'], format="%Y-%m-%d %H:%M:%S")

# since there is a clear pattern seen in "time" I create "hour"
data['hour'] = data['datetime'].dt.hour
data['weekday'] = data['datetime'].dt.weekday
data['is_weekend'] = data['weekday'] >= 5

data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)


In [ ]:
# How is the behaviour of each day?

weekday_counts = data.groupby('weekday').size()

days_per_weekday = data.groupby('weekday')['date'].nunique()
normalized_weekday = weekday_counts / days_per_weekday
print(normalized_weekday)

normalized_weekday.plot(
    kind='line',
    marker='o',
    figsize=(12,5),
    title="Average Activity per Weekday (Normalized)",
    grid=True
)



There is no clear difference between weekdays in this plot; the number of bookings appears to be approximately the same each day 

In [ ]:
# Is the behaviour changing in the weekend? 

avg_activity = (
    data.groupby('is_weekend')
        .size()
        / data.groupby('is_weekend')['date'].nunique()
)
print(avg_activity)

avg_activity.plot(
    kind='bar',
    title='Average Activity per Day (Weekend vs Weekday)',
    grid=True
)


After normalizing by the number of days, weekend and weekday activity levels are comparable, indicating no strong weekend effect

In [ ]:
data[["hour_sin", "hour_cos"]].describe()

Expected: min ≈ -1, max ≈ 1, mean ≈ 0, perfect

To capture the cyclical nature of time, the hour of day was encoded using sine and cosine transformations. This representation preserves the circular structure of time, ensuring that adjacent hours such as 23:00 and 00:00 are represented as similar values.

#### Categorical

In [ ]:
pd.set_option("display.max_rows", None)
cat_cols = df_typed.select_dtypes(include="category").columns


In [ ]:
for col in cat_cols:
    print(f"\n=== {col} ===")
    print(df_typed[col].value_counts(dropna=False))


### String

In [ ]:
string_cols = df_typed.select_dtypes(include="string[pyarrow]").columns


In [ ]:
for col in string_cols:
    print(f"\n=== {col} ===")
    print(df_typed[col].value_counts(dropna=False))

### Numerical

In [ ]:
df_typed.describe()

In [ ]:
df_typed.hist(figsize=(20, 15), bins = 100)